# Pruebas de integración: Robot médico hospitalario

Este notebook une el entorno, Value Iteration, Q-Learning y las métricas.  
La idea es correr todo bajo las mismas condiciones y guardar tablas/gráficas en `results/`.


## 1. Configuración inicial

Detecta la raíz del repositorio y permite importar archivos desde `src/`.


In [ ]:
from pathlib import Path
import sys
import random
import subprocess

import pandas as pd
import matplotlib.pyplot as plt


# Semilla para que las pruebas sean más comparables.
random.seed(42)


Raíz del proyecto: c:\Users\viank\OneDrive\Desktop\S1_2026\IA_Proyecto_Final


## 2. Importar módulos del proyecto

In [17]:
from src.environment import HospitalEnvironment, Action, START_POS, GOAL_POS, MAX_BATTERY
from src.value_iteration import ValueIterationAgent
from src.q_learning import QLearningAgent
from src.metrics import evaluate_agent, compare_agents, save_results_table


## 3. Prueba rápida del entorno

Esto confirma que el ambiente se puede crear, reiniciar y consultar antes de entrenar modelos.


In [18]:
env = HospitalEnvironment()
initial_state = env.reset()

print("Estado inicial:", initial_state)
print("Posición inicial esperada:", START_POS)
print("Destino:", GOAL_POS)
print("Batería máxima:", MAX_BATTERY)
print("Acciones posibles al inicio:", [a.name for a in env.get_possible_actions(initial_state)])


Estado inicial: (1, 1, 30)
Posición inicial esperada: (1, 1)
Destino: (10, 14)
Batería máxima: 30
Acciones posibles al inicio: ['UP', 'DOWN', 'LEFT', 'RIGHT', 'STAY']


## 4. Ejecutar pruebas unitarias

In [19]:
tests_dir = ROOT / "tests"
if tests_dir.exists():
    result = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", str(tests_dir), "-v"],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print(result.stderr)
else:
    print("No se encontró la carpeta tests/. Se omite esta verificación.")



test_battery_decreases_on_movement (test_environment.TestBatteryDrain.test_battery_decreases_on_movement) ... ok
test_battery_decreases_on_stay (test_environment.TestBatteryDrain.test_battery_decreases_on_stay) ... ok
test_battery_zero_triggers_done (test_environment.TestBatteryDrain.test_battery_zero_triggers_done) ... ok
test_charge_does_not_exceed_max_battery (test_environment.TestCharging.test_charge_does_not_exceed_max_battery) ... ok
test_charge_does_not_move_robot (test_environment.TestCharging.test_charge_does_not_move_robot) ... ok
test_charge_increases_battery_at_station (test_environment.TestCharging.test_charge_increases_battery_at_station) ... ok
test_charge_outside_station_does_nothing (test_environment.TestCharging.test_charge_outside_station_does_nothing) ... ok
test_battery_range_correct (test_environment.TestGetAllStates.test_battery_range_correct) ... ok
test_no_wall_cells_in_states (test_environment.TestGetAllStates.test_no_wall_cells_in_states) ... ok
test_total_c

## 5. Ejecutar Value Iteration

Value Iteration usa el modelo del entorno para calcular una política.  
En este proyecto representa el enfoque basado en MDP con modelo conocido.


In [20]:
random.seed(42)

env_vi = HospitalEnvironment()
vi_agent = ValueIterationAgent(env_vi, gamma=0.9, theta=1e-3)
vi_agent.run(verbose=True)

print("Iteraciones hasta converger:", vi_agent.iterations)
print("Acción recomendada al inicio:", vi_agent.get_action(env_vi.reset()).name)


Iter   10  Δ = 19.371024
Iter   20  Δ = 2.927521
Iter   30  Δ = 0.306108
Iter   40  Δ = 0.083290
Iter   50  Δ = 0.001963
Convergió en 52 iteraciones (Δ = 0.000942)
Iteraciones hasta converger: 52
Acción recomendada al inicio: DOWN


## 6. Entrenar Q-Learning

Q-Learning aprende por experiencia.  
Con 1,000 episodios puede salir muy bajo porque el entorno es grande, estocástico y la recompensa positiva está lejos del inicio. Para una comparación más justa, se usa un entrenamiento más largo.

Sugerencia para entrega: usar entre `50_000` y `100_000` episodios si la computadora lo soporta.


In [21]:
random.seed(42)

env_ql = HospitalEnvironment()

ql_agent = QLearningAgent(
    env_ql,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.3,
)

QL_EPISODES = 50_000

ql_agent.train(episodes=QL_EPISODES, max_steps=200)

print("Episodios entrenados:", len(ql_agent.rewards_per_episode))
print("Estados aprendidos en Q-table:", len(ql_agent.Q))
print("Acción recomendada al inicio:", ql_agent.get_action(env_ql.reset()).name)


TypeError: QLearningAgent.train() got an unexpected keyword argument 'max_steps'

## 7. Evaluar ambos agentes

La evaluación se hace con `epsilon = 0` para Q-Learning, así se evalúa la política aprendida sin exploración aleatoria.


In [ ]:
random.seed(123)

EVAL_EPISODES = 100

results_vi = evaluate_agent(
    env_vi,
    vi_agent,
    episodes=EVAL_EPISODES,
    max_steps=200,
    agent_name="Value Iteration"
)

results_ql = evaluate_agent(
    env_ql,
    ql_agent,
    episodes=EVAL_EPISODES,
    max_steps=200,
    agent_name="Q-Learning"
)

comparison = compare_agents(results_vi, results_ql)
comparison


## 8. Guardar tablas de resultados

In [ ]:
tables_dir = ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

save_results_table(comparison, tables_dir / "metricas_comparativas.csv")

results_vi["details"].to_csv(tables_dir / "detalle_value_iteration.csv", index=False, encoding="utf-8-sig")
results_ql["details"].to_csv(tables_dir / "detalle_q_learning.csv", index=False, encoding="utf-8-sig")

print("Tablas guardadas en:", tables_dir)


## 9. Gráficas comparativas

Estas gráficas son las más útiles para el informe y la presentación.


In [ ]:
plots_dir = ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

def save_bar_chart(column, title, ylabel, filename):
    data = comparison.copy()
    plt.figure(figsize=(8, 5))
    plt.bar(data["Agente"], data[column].fillna(0))
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xlabel("Agente")
    plt.savefig(plots_dir / filename, bbox_inches="tight")
    plt.show()

save_bar_chart("% entregas exitosas", "Porcentaje de entregas exitosas por agente", "% entregas exitosas", "entregas_exitosas.png")
save_bar_chart("Recompensa acumulada promedio", "Recompensa acumulada promedio por agente", "Recompensa promedio", "recompensa_promedio.png")
save_bar_chart("Consumo promedio de batería", "Consumo promedio de batería por agente", "Batería consumida", "consumo_bateria.png")
save_bar_chart("Fallos de misión", "Fallos de misión por agente", "Cantidad de fallos", "fallos_mision.png")
save_bar_chart("Tiempo promedio de entrega", "Tiempo promedio de entrega por agente", "Pasos promedio", "tiempo_entrega.png")


## 10. Curva de aprendizaje de Q-Learning

Esta gráfica muestra cómo evolucionó la recompensa acumulada durante el entrenamiento.


In [ ]:
rewards = pd.Series(ql_agent.rewards_per_episode)
rolling_rewards = rewards.rolling(window=500, min_periods=1).mean()

plt.figure(figsize=(10, 5))
plt.plot(rewards, alpha=0.25, label="Recompensa por episodio")
plt.plot(rolling_rewards, label="Promedio móvil 500 episodios")
plt.title("Curva de aprendizaje - Q-Learning")
plt.xlabel("Episodio")
plt.ylabel("Recompensa acumulada")
plt.legend()
plt.savefig(plots_dir / "q_learning_rewards.png", bbox_inches="tight")
plt.show()


In [ ]:
tables_dir = ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

save_results_table(comparison, tables_dir / "metricas_comparativas.csv")

results_vi["details"].to_csv(
    tables_dir / "detalle_value_iteration.csv",
    index=False,
    encoding="utf-8-sig"
)

results_ql["details"].to_csv(
    tables_dir / "detalle_q_learning.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tablas guardadas en:", tables_dir)

In [ ]:
plots_dir = ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

def save_bar_chart(column, title, ylabel, filename):
    data = comparison.copy()

    plt.figure(figsize=(8, 5))
    plt.bar(data["Agente"], data[column].fillna(0))
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xlabel("Agente")
    plt.savefig(plots_dir / filename, bbox_inches="tight")
    plt.show()

save_bar_chart(
    "% entregas exitosas",
    "Porcentaje de entregas exitosas por agente",
    "% entregas exitosas",
    "entregas_exitosas.png"
)

save_bar_chart(
    "Recompensa acumulada promedio",
    "Recompensa acumulada promedio por agente",
    "Recompensa promedio",
    "recompensa_promedio.png"
)

save_bar_chart(
    "Consumo promedio de batería",
    "Consumo promedio de batería por agente",
    "Batería consumida",
    "consumo_bateria.png"
)

save_bar_chart(
    "Fallos de misión",
    "Fallos de misión por agente",
    "Cantidad de fallos",
    "fallos_mision.png"
)

save_bar_chart(
    "Tiempo promedio de entrega",
    "Tiempo promedio de entrega por agente",
    "Pasos promedio",
    "tiempo_entrega.png"
)

In [ ]:
rewards = pd.Series(ql_agent.rewards_per_episode)
rolling_rewards = rewards.rolling(window=500, min_periods=1).mean()

plt.figure(figsize=(10, 5))
plt.plot(rewards, alpha=0.25, label="Recompensa por episodio")
plt.plot(rolling_rewards, label="Promedio móvil 500 episodios")
plt.title("Curva de aprendizaje - Q-Learning")
plt.xlabel("Episodio")
plt.ylabel("Recompensa acumulada")
plt.legend()
plt.savefig(plots_dir / "q_learning_rewards.png", bbox_inches="tight")
plt.show()

## 11. Interpretación breve para el informe

Usa esta interpretación como base. Ajusta los números según la tabla que te salga al correr el notebook.


In [ ]:
vi_success = comparison.loc[comparison["Agente"] == "Value Iteration", "% entregas exitosas"].iloc[0]
ql_success = comparison.loc[comparison["Agente"] == "Q-Learning", "% entregas exitosas"].iloc[0]
vi_reward = comparison.loc[comparison["Agente"] == "Value Iteration", "Recompensa acumulada promedio"].iloc[0]
ql_reward = comparison.loc[comparison["Agente"] == "Q-Learning", "Recompensa acumulada promedio"].iloc[0]

print("Interpretación sugerida:")
print(f"Value Iteration obtuvo {vi_success:.1f}% de entregas exitosas y una recompensa promedio de {vi_reward:.2f}.")
print(f"Q-Learning obtuvo {ql_success:.1f}% de entregas exitosas y una recompensa promedio de {ql_reward:.2f}.")
print("Esto permite comparar un método basado en modelo contra un método que aprende por experiencia.")
print("Si Q-Learning queda por debajo de Value Iteration, es coherente con el problema: Value Iteration conoce el modelo del entorno, mientras que Q-Learning debe aprenderlo mediante exploración.")
